In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Construcción de Datamart Gold: Customer 360
Este notebook toma los datos transaccionales y dimensionales de la capa Silver para generar una vista analítica avanzada

In [0]:
# 1. Lectura de tablas Silver
df_trans = spark.table('castor.silver.slv_fact_transactions')
df_users = spark.table('castor.silver.slv_dim_users')

In [0]:
# 2. RFM (Recency, Frequency, Monetary) y KPIs básicos
# Definimos la 'fecha actual' (el máximo de las transacciones) para calcular la recencia
max_date_df = df_trans.select(F.max('fecha_transaccion').alias('max_date'))
max_date = max_date_df.collect()[0]['max_date']

rfm_df = df_trans.groupBy('id_usuario').agg(
    F.datediff(F.lit(max_date), F.max('fecha_transaccion')).alias('recency_dias'),
    F.count('id_transaccion').alias('frequency_txs'),
    F.sum('monto').alias('monetary_total'),
    F.avg('monto').alias('ticket_promedio'),
    F.min('fecha_transaccion').alias('primera_compra'),
    F.max('fecha_transaccion').alias('ultima_compra')
)

In [0]:
# 3. Encontrar la categoría favorita de cada usuario (donde gastó más dinero)
window_cat = Window.partitionBy('id_usuario').orderBy(F.col('gasto_categoria').desc())

cat_df = df_trans.groupBy('id_usuario', 'categoria').agg(F.sum('monto').alias('gasto_categoria'))
fav_cat_df = cat_df.withColumn('rn', F.row_number().over(window_cat)) \
                   .filter(F.col('rn') == 1) \
                   .select('id_usuario', F.col('categoria').alias('categoria_favorita'))

In [0]:
# 4. Enriquecimiento Demográfico (Segmentación de Edad y Antigüedad)
df_users_enriched = df_users.withColumn(
    'rango_edad',
    F.when(F.col('edad') < 25, '18-24')
     .when((F.col('edad') >= 25) & (F.col('edad') <= 35), '25-35')
     .when((F.col('edad') > 35) & (F.col('edad') <= 50), '36-50')
     .otherwise('51+')
).withColumn(
    'antiguedad_dias',
    F.datediff(F.lit(max_date), F.col('fecha_registro'))
)

In [0]:
# 5. Join Final: Construcción del Datamart Customer 360
# Unimos RFM, Categoría Favorita y Datos Demográficos
df_customer_360 = df_users_enriched \
    .join(rfm_df, on='id_usuario', how='left') \
    .join(fav_cat_df, on='id_usuario', how='left')

# Llenamos nulos para usuarios que se registraron pero aún no han comprado (Frequency = 0)
df_customer_360 = df_customer_360.fillna({
    'frequency_txs': 0,
    'monetary_total': 0.0,
    'ticket_promedio': 0.0,
    'categoria_favorita': 'Sin compras'
})

display(df_customer_360.limit(10))

In [0]:
# 6. Guardar en la capa Gold
# Este tablón es perfecto para Tableau. No particionamos por mes/año de la transacción 
# porque es un snapshot a nivel usuario, lo que acelera los dashboard analíticos.
df_customer_360.write.mode('overwrite').saveAsTable('castor.gold.gld_customer_360')